# Sample Ticket Step 1 to Step 2 Timing

This notebook reads ticket numbers from `data/samples/sample_tickets.txt`, pulls their action histories from the local SQLite database, reconstructs compressed authority steps, summarizes the time taken to move from step 1 to step 2, exports the resulting tables to Excel, and saves polished figures to disk.

It also mirrors the outlier-handling logic used in `1.05-ym-time-to-assignment.ipynb`: Twitter complaints use the first substantive handoff directly, while non-Twitter complaints skip the initial complaint-registration event before measuring the next handoff.

In [ ]:
from pathlib import Path
import sqlite3
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "app").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate repo root containing 'app/' and 'data/'.")


REPO_ROOT = find_repo_root()
DB_PATH = REPO_ROOT / "data" / "raw" / "grievance.db"
TICKET_FILE = REPO_ROOT / "data" / "samples" / "sample_tickets.txt"
OUTPUT_DIR = REPO_ROOT / "output" / "sample_ticket_step12"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXCEL_PATH = OUTPUT_DIR / "sample_ticket_step12_summary.xlsx"
FIG_HIST_PATH = OUTPUT_DIR / "step12_gap_histogram_adjusted.png"
FIG_PAIR_PATH = OUTPUT_DIR / "step12_pair_medians_adjusted.png"
FIG_MODE_PATH = OUTPUT_DIR / "step12_mode_medians_adjusted.png"

# UChicago-style defaults borrowed from 1.04
MAROON = "#800000"
MAROON_MID = "#A84040"
MAROON_LIGHT = "#D4A0A0"
NEAR_BLACK = "#1A1A1A"
GREY_AXIS = "#AAAAAA"
GREY_GRID = "#E8E8E8"
MID_GREY = "#6D737A"
GOLD = "#C48A00"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GREY_AXIS,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": GREY_GRID,
    "grid.linewidth": 0.6,
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlecolor": NEAR_BLACK,
    "axes.labelcolor": NEAR_BLACK,
    "xtick.color": "#555555",
    "ytick.color": "#555555",
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})


def save_polished(fig, path: Path):
    fig.savefig(path, dpi=300, bbox_inches="tight", pad_inches=0.16, facecolor="white")
    print(f"Saved: {path}")


print(f"Repo root: {REPO_ROOT}")
print(f"DB path: {DB_PATH}")
print(f"Ticket file: {TICKET_FILE}")
print(f"Output dir: {OUTPUT_DIR}")

## Outlier handling mirrored from `1.05-ym-time-to-assignment.ipynb`

That notebook does three things:

1. It first checks the naive `created_on -> first action` gap and finds many non-Twitter complaints with a zero-hour first action.
2. It ranks action-history rows within each ticket by time.
3. It then treats the first **meaningful** action differently by mode:
   - `Twitter`: use `action_rank == 1`
   - `non-Twitter`: use `action_rank == 2`

For this notebook, the analogous adjustment is:

- `Twitter`: measure compressed step `1 -> 2`
- `non-Twitter`: skip the initial complaint-registration step and measure compressed step `2 -> 3`

The adjusted version is the primary output. The raw `1 -> 2` calculation is also kept for auditability.

In [ ]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"DB not found: {DB_PATH}")
if not TICKET_FILE.exists():
    raise FileNotFoundError(f"Ticket file not found: {TICKET_FILE}")

ticket_nos = [line.strip() for line in TICKET_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
ticket_nos = list(dict.fromkeys(ticket_nos))

print(f"Loaded {len(ticket_nos):,} unique tickets from file")
ticket_nos[:10]

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.execute("DROP TABLE IF EXISTS temp.selected_tickets")
conn.execute("CREATE TEMP TABLE selected_tickets (ticket_no TEXT PRIMARY KEY)")
conn.executemany(
    "INSERT OR IGNORE INTO selected_tickets(ticket_no) VALUES (?)",
    [(ticket_no,) for ticket_no in ticket_nos],
)

actions = pd.read_sql_query(
    """
    SELECT a.*
    FROM action_history AS a
    INNER JOIN selected_tickets AS t ON a.ticket_no = t.ticket_no
    ORDER BY a.ticket_no, a.action_taken_date, a.id
    """,
    conn,
)

complaints = pd.read_sql_query(
    """
    SELECT c.ticket_no, c.created_on, c.resolved_on, c.status, c.subcategory, c.mode
    FROM complaints AS c
    INNER JOIN selected_tickets AS t ON c.ticket_no = t.ticket_no
    """,
    conn,
)

print(f"Loaded {len(actions):,} action-history rows")
print(f"Loaded {complaints['ticket_no'].nunique():,} complaint rows")
actions.head()

In [ ]:
actions["action_taken_date"] = pd.to_datetime(actions["action_taken_date"], errors="coerce")
complaints["created_on"] = pd.to_datetime(complaints["created_on"], errors="coerce")
complaints["resolved_on"] = pd.to_datetime(complaints["resolved_on"], errors="coerce")
complaints["mode"] = complaints["mode"].fillna("Unknown").astype(str).str.strip()


def authority_from_status(row: pd.Series) -> str:
    status = "" if pd.isna(row.get("complaint_status_with_authority")) else str(row["complaint_status_with_authority"]).strip()
    taken_by = "" if pd.isna(row.get("action_taken_by")) else str(row["action_taken_by"]).strip()
    if " - " in status:
        return status.split(" - ", 1)[1].strip()
    return taken_by


def clean_authority(value: str) -> str:
    s = "" if pd.isna(value) else str(value).strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s*,\s*", ", ", s)
    if "," in s:
        s = s.split(",", 1)[0].strip()
    s = s.upper()
    s = s.replace("COLLECTOR & DM", "COLLECTOR")
    s = s.replace("COLLECTOR AND DM", "COLLECTOR")
    s = s.replace("B.D.O", "BDO")
    s = s.replace("BLOCK DEVELOPMENT OFFICER", "BDO")
    s = s.replace("CHIEF MINISTER", "CM")
    s = s.replace("CMO", "CM")
    s = re.sub(r"\s{2,}", " ", s).strip(" -_/,")
    return s


actions["authority_raw"] = actions.apply(authority_from_status, axis=1)
actions["authority_clean"] = actions["authority_raw"].apply(clean_authority)

actions = actions[
    actions["ticket_no"].notna()
    & actions["action_taken_date"].notna()
    & (actions["authority_clean"] != "")
].copy()

actions = actions.sort_values(["ticket_no", "action_taken_date", "id"]).reset_index(drop=True)
actions = actions.merge(complaints[["ticket_no", "created_on", "mode"]], on="ticket_no", how="left")

actions[["ticket_no", "action_taken_date", "authority_clean", "mode", "complaint_status_with_authority"]].head(10)

In [ ]:
# Replicate the 1.05 diagnostic: naive first action is often zero-hour for non-Twitter complaints.
ranked_actions = actions.sort_values(["ticket_no", "action_taken_date", "id"]).copy()
ranked_actions["action_rank"] = ranked_actions.groupby("ticket_no").cumcount() + 1
ranked_actions["n_actions"] = ranked_actions.groupby("ticket_no")["ticket_no"].transform("size")

naive_first_action = ranked_actions[ranked_actions["action_rank"] == 1].copy()
naive_first_action["delta_hours_first_action"] = (
    (naive_first_action["action_taken_date"] - naive_first_action["created_on"]).dt.total_seconds() / 3600
)

first_action_diagnostic = (
    naive_first_action.groupby("mode", dropna=False)
    .agg(
        tickets=("ticket_no", "count"),
        mean_hours=("delta_hours_first_action", "mean"),
        median_hours=("delta_hours_first_action", "median"),
        p90_hours=("delta_hours_first_action", lambda s: s.quantile(0.90)),
        zero_hour_share=("delta_hours_first_action", lambda s: (s.fillna(-1) == 0).mean()),
    )
    .reset_index()
    .sort_values("tickets", ascending=False)
)
first_action_diagnostic["zero_hour_share"] = (first_action_diagnostic["zero_hour_share"] * 100).round(2)
first_action_diagnostic = first_action_diagnostic.round(2)

display(first_action_diagnostic.head(15))

In [ ]:
def compress_ticket_steps(group: pd.DataFrame) -> pd.DataFrame:
    rows = []
    previous_authority = None

    for _, row in group.sort_values(["action_taken_date", "id"]).iterrows():
        authority = row["authority_clean"]
        if authority != previous_authority:
            rows.append(
                {
                    "ticket_no": row["ticket_no"],
                    "step_no": len(rows) + 1,
                    "source_step_no": len(rows) + 1,
                    "authority_clean": authority,
                    "step_start": row["action_taken_date"],
                }
            )
            previous_authority = authority

    return pd.DataFrame(rows)


def build_gap_frame(step_frame: pd.DataFrame, step_index_col: str, complaints_meta: pd.DataFrame) -> pd.DataFrame:
    subset = step_frame[step_frame[step_index_col].isin([1, 2])].copy()
    wide = subset.pivot(index="ticket_no", columns=step_index_col, values=["authority_clean", "step_start", "source_step_no"])
    wide.columns = [f"{name}_{int(step_no)}" for name, step_no in wide.columns]
    wide = wide.reset_index().rename(
        columns={
            "authority_clean_1": "step_1_name",
            "authority_clean_2": "step_2_name",
            "step_start_1": "step_1_start",
            "step_start_2": "step_2_start",
            "source_step_no_1": "source_step_1_no",
            "source_step_no_2": "source_step_2_no",
        }
    )
    wide["gap_1_2_days"] = (
        (wide["step_2_start"] - wide["step_1_start"]).dt.total_seconds() / (24 * 3600)
    )
    wide["pair_1_2"] = wide["step_1_name"].fillna("") + " -> " + wide["step_2_name"].fillna("")
    wide = wide.merge(complaints_meta, on="ticket_no", how="left")
    wide = wide[
        wide["step_1_start"].notna()
        & wide["step_2_start"].notna()
        & wide["gap_1_2_days"].notna()
        & (wide["gap_1_2_days"] >= 0)
    ].copy()
    return wide


def summarize_gap_frame(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    s = frame["gap_1_2_days"].dropna()
    return pd.DataFrame(
        {
            "definition": [label],
            "tickets": [len(frame)],
            "mean_days": [s.mean()],
            "median_days": [s.median()],
            "std_days": [s.std()],
            "min_days": [s.min()],
            "p10_days": [s.quantile(0.10)],
            "p25_days": [s.quantile(0.25)],
            "p75_days": [s.quantile(0.75)],
            "p90_days": [s.quantile(0.90)],
            "p95_days": [s.quantile(0.95)],
            "max_days": [s.max()],
        }
    ).round(2)


def summarize_pairs(frame: pd.DataFrame) -> pd.DataFrame:
    out = (
        frame.groupby("pair_1_2", dropna=False)
        .agg(
            tickets=("ticket_no", "count"),
            mean_days=("gap_1_2_days", "mean"),
            median_days=("gap_1_2_days", "median"),
            std_days=("gap_1_2_days", "std"),
            p10_days=("gap_1_2_days", lambda s: s.quantile(0.10)),
            p25_days=("gap_1_2_days", lambda s: s.quantile(0.25)),
            p75_days=("gap_1_2_days", lambda s: s.quantile(0.75)),
            p90_days=("gap_1_2_days", lambda s: s.quantile(0.90)),
            p95_days=("gap_1_2_days", lambda s: s.quantile(0.95)),
            min_days=("gap_1_2_days", "min"),
            max_days=("gap_1_2_days", "max"),
        )
        .reset_index()
    )
    out["share_pct"] = (out["tickets"] / len(frame) * 100).round(2)
    return out.sort_values(["tickets", "median_days"], ascending=[False, False]).reset_index(drop=True).round(2)


step_rows = (
    actions.groupby("ticket_no", group_keys=False)
    .apply(compress_ticket_steps)
    .reset_index(drop=True)
)

complaints_meta = complaints[["ticket_no", "created_on", "resolved_on", "status", "subcategory", "mode"]].copy()
step_rows = step_rows.merge(complaints_meta[["ticket_no", "mode"]], on="ticket_no", how="left")

raw_step12_wide = build_gap_frame(step_rows, "step_no", complaints_meta)

adjusted_candidates = step_rows[
    ((step_rows["mode"] == "Twitter") & (step_rows["step_no"].isin([1, 2])))
    | ((step_rows["mode"] != "Twitter") & (step_rows["step_no"].isin([2, 3])))
].copy()
adjusted_candidates["analysis_step_no"] = np.where(
    adjusted_candidates["mode"] == "Twitter",
    adjusted_candidates["step_no"],
    adjusted_candidates["step_no"] - 1,
)
adjusted_candidates["selection_rule"] = np.where(
    adjusted_candidates["mode"] == "Twitter",
    "Twitter: raw steps 1 -> 2",
    "Non-Twitter: raw steps 2 -> 3",
)

adjusted_step12_wide = build_gap_frame(adjusted_candidates, "analysis_step_no", complaints_meta)
adjusted_step12_wide["selection_rule"] = np.where(
    adjusted_step12_wide["mode"] == "Twitter",
    "Twitter: raw steps 1 -> 2",
    "Non-Twitter: raw steps 2 -> 3",
)

missing_adjusted_ticket_df = pd.DataFrame({"ticket_no": ticket_nos})
missing_adjusted_ticket_df = missing_adjusted_ticket_df[~missing_adjusted_ticket_df["ticket_no"].isin(adjusted_step12_wide["ticket_no"])]

coverage_summary_df = pd.DataFrame(
    {
        "metric": [
            "tickets_in_file",
            "tickets_found_in_complaints",
            "tickets_with_action_history",
            "tickets_with_raw_1_to_2_gap",
            "tickets_with_adjusted_gap",
            "tickets_missing_adjusted_gap",
        ],
        "value": [
            len(ticket_nos),
            complaints["ticket_no"].nunique(),
            actions["ticket_no"].nunique(),
            len(raw_step12_wide),
            len(adjusted_step12_wide),
            len(missing_adjusted_ticket_df),
        ],
    }
)

overall_summary_df = pd.concat(
    [
        summarize_gap_frame(raw_step12_wide, "Raw compressed steps 1 -> 2"),
        summarize_gap_frame(adjusted_step12_wide, "Mode-adjusted: Twitter 1 -> 2, non-Twitter 2 -> 3"),
    ],
    ignore_index=True,
)

pair_summary_raw = summarize_pairs(raw_step12_wide)
pair_summary_adjusted = summarize_pairs(adjusted_step12_wide)

mode_summary_adjusted = (
    adjusted_step12_wide.groupby("mode", dropna=False)
    .agg(
        tickets=("ticket_no", "count"),
        mean_days=("gap_1_2_days", "mean"),
        median_days=("gap_1_2_days", "median"),
        std_days=("gap_1_2_days", "std"),
        p10_days=("gap_1_2_days", lambda s: s.quantile(0.10)),
        p25_days=("gap_1_2_days", lambda s: s.quantile(0.25)),
        p75_days=("gap_1_2_days", lambda s: s.quantile(0.75)),
        p90_days=("gap_1_2_days", lambda s: s.quantile(0.90)),
        p95_days=("gap_1_2_days", lambda s: s.quantile(0.95)),
        min_days=("gap_1_2_days", "min"),
        max_days=("gap_1_2_days", "max"),
    )
    .reset_index()
)
mode_summary_adjusted["share_pct"] = (mode_summary_adjusted["tickets"] / len(adjusted_step12_wide) * 100).round(2)
mode_summary_adjusted = mode_summary_adjusted.sort_values(["tickets", "median_days"], ascending=[False, False]).reset_index(drop=True).round(2)

mode_pair_summary_adjusted = (
    adjusted_step12_wide.groupby(["mode", "pair_1_2"], dropna=False)
    .agg(
        tickets=("ticket_no", "count"),
        mean_days=("gap_1_2_days", "mean"),
        median_days=("gap_1_2_days", "median"),
        p90_days=("gap_1_2_days", lambda s: s.quantile(0.90)),
        p95_days=("gap_1_2_days", lambda s: s.quantile(0.95)),
    )
    .reset_index()
    .sort_values(["mode", "tickets", "median_days"], ascending=[True, False, False])
    .reset_index(drop=True)
    .round(2)
)

top_gaps_adjusted_df = adjusted_step12_wide.sort_values("gap_1_2_days", ascending=False)[
    [
        "ticket_no",
        "mode",
        "selection_rule",
        "source_step_1_no",
        "source_step_2_no",
        "pair_1_2",
        "step_1_name",
        "step_2_name",
        "step_1_start",
        "step_2_start",
        "gap_1_2_days",
        "status",
        "subcategory",
    ]
].head(50)

display(coverage_summary_df)
display(overall_summary_df)
display(pair_summary_adjusted.head(20))
display(mode_summary_adjusted.head(20))

In [ ]:
plot_series = adjusted_step12_wide["gap_1_2_days"].dropna()
if plot_series.empty:
    raise ValueError("No usable adjusted step 1 -> step 2 gaps found for the supplied tickets.")

plot_cap = plot_series.quantile(0.99)
plot_data = plot_series[plot_series <= plot_cap]

fig, ax = plt.subplots(figsize=(10.5, 5.8))
ax.hist(plot_data, bins=40, color=MAROON, alpha=0.88, edgecolor="white", linewidth=0.3)
ax.axvline(plot_series.median(), color=NEAR_BLACK, linewidth=1.5, linestyle="--")
ax.axvline(plot_series.mean(), color=GOLD, linewidth=1.5, linestyle=":")

ax.set_title("Distribution of time taken from complaint receipt to routing to first step", fontsize=12, color=NEAR_BLACK)
ax.set_xlabel("Num. days")
ax.set_ylabel("No. of tickets")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.spines["left"].set_color(GREY_AXIS)
ax.spines["bottom"].set_color(GREY_AXIS)

ymax = ax.get_ylim()[1]
ax.text(
    plot_series.median() + max(plot_series.median() * 0.02, 0.3),
    ymax * 0.93,
    f"Median: {plot_series.median():.2f}d",
    fontsize=9,
    color=NEAR_BLACK,
)
ax.text(
    plot_series.mean() + max(plot_series.mean() * 0.02, 0.3),
    ymax * 0.82,
    f"Mean: {plot_series.mean():.2f}d",
    fontsize=9,
    color=GOLD,
)
ax.text(
    0.98,
    0.97,
    f"n = {len(plot_series):,}",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8,
    color="#777777",
)

fig.text(
    0.5,
    -0.02,
    f"Twitter uses compressed steps 1 -> 2; non-Twitter uses compressed steps 2 -> 3. Histogram clipped at p99 = {plot_cap:.2f} days for display.",
    ha="center",
    fontsize=8,
    color="#777777",
    style="italic",
)

plt.tight_layout()
save_polished(fig, FIG_HIST_PATH)
plt.show()

In [ ]:
top_pairs = pair_summary_adjusted.head(12).iloc[::-1].copy()

fig, ax = plt.subplots(figsize=(11.2, 6.6))
fig.subplots_adjust(left=0.35, right=0.96, top=0.92, bottom=0.18)

bars = ax.barh(top_pairs["pair_1_2"], top_pairs["median_days"], color=MAROON_MID, height=0.64, zorder=3)
ax.scatter(top_pairs["mean_days"], top_pairs["pair_1_2"], color=GOLD, s=70, zorder=4, label="Mean")

max_x = float(top_pairs[["median_days", "mean_days"]].max().max()) if len(top_pairs) else 1.0
for bar, (_, row) in zip(bars, top_pairs.iterrows()):
    ax.text(
        bar.get_width() + max(max_x * 0.015, 0.2),
        bar.get_y() + bar.get_height() / 2,
        f"med {row['median_days']:.1f}d | n={int(row['tickets']):,}",
        va="center",
        ha="left",
        fontsize=9,
        color=NEAR_BLACK,
    )

ax.set_title("Most common mode-adjusted step 1 to step 2 handoff pairs", fontsize=12, color=NEAR_BLACK)
ax.set_xlabel("Median days")
ax.set_ylabel("Step pair")
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.spines["left"].set_color(GREY_AXIS)
ax.spines["bottom"].set_color(GREY_AXIS)
ax.legend(frameon=False, loc="lower right")

fig.text(
    0.5,
    -0.02,
    "Bars show median days; gold markers show mean days. Pairs are sorted by ticket volume, then median duration.",
    ha="center",
    fontsize=8,
    color="#777777",
    style="italic",
)

save_polished(fig, FIG_PAIR_PATH)
plt.show()

In [ ]:
top_gaps_adjusted_df.head(25)

In [ ]:
mode_plot_df = mode_summary_adjusted.head(12).iloc[::-1].copy()

fig, ax = plt.subplots(figsize=(10.6, 6.2))
fig.subplots_adjust(left=0.28, right=0.96, top=0.92, bottom=0.18)

bars = ax.barh(mode_plot_df["mode"], mode_plot_df["median_days"], color=MAROON, height=0.62, zorder=3)
ax.scatter(mode_plot_df["mean_days"], mode_plot_df["mode"], color=GOLD, s=70, zorder=4, label="Mean")

max_x = float(mode_plot_df[["median_days", "mean_days"]].max().max()) if len(mode_plot_df) else 1.0
for bar, (_, row) in zip(bars, mode_plot_df.iterrows()):
    ax.text(
        bar.get_width() + max(max_x * 0.015, 0.2),
        bar.get_y() + bar.get_height() / 2,
        f"med {row['median_days']:.1f}d | n={int(row['tickets']):,}",
        va="center",
        ha="left",
        fontsize=9,
        color=NEAR_BLACK,
    )

ax.set_title("Mode-wise comparison of adjusted routing time", fontsize=12, color=NEAR_BLACK)
ax.set_xlabel("Median days")
ax.set_ylabel("Mode")
ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.spines["left"].set_color(GREY_AXIS)
ax.spines["bottom"].set_color(GREY_AXIS)
ax.legend(frameon=False, loc="lower right")

fig.text(
    0.5,
    -0.02,
    "Twitter uses raw compressed steps 1 -> 2; non-Twitter modes use raw compressed steps 2 -> 3. Bars show medians; gold markers show means.",
    ha="center",
    fontsize=8,
    color="#777777",
    style="italic",
)

save_polished(fig, FIG_MODE_PATH)
plt.show()

In [ ]:
methodology_df = pd.DataFrame(
    {
        "step": [1, 2, 3, 4],
        "description": [
            "Read ticket list and pull matching complaints + action_history from SQLite",
            "Clean action authorities and compress consecutive duplicate authorities into steps",
            "Apply 1.05 adjustment: Twitter uses raw compressed steps 1 -> 2; non-Twitter uses raw compressed steps 2 -> 3",
            "Summarize adjusted gap distribution, export tables to Excel, and save polished figures",
        ],
    }
)

figure_manifest_df = pd.DataFrame(
    {
        "figure_name": ["step12_gap_histogram_adjusted", "step12_pair_medians_adjusted", "step12_mode_medians_adjusted"],
        "path": [str(FIG_HIST_PATH), str(FIG_PAIR_PATH), str(FIG_MODE_PATH)],
    }
)

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    methodology_df.to_excel(writer, sheet_name="methodology", index=False)
    coverage_summary_df.to_excel(writer, sheet_name="coverage_summary", index=False)
    overall_summary_df.to_excel(writer, sheet_name="overall_summary", index=False)
    first_action_diagnostic.to_excel(writer, sheet_name="first_action_diag", index=False)
    pair_summary_adjusted.to_excel(writer, sheet_name="pair_summary_adjusted", index=False)
    pair_summary_raw.to_excel(writer, sheet_name="pair_summary_raw", index=False)
    mode_summary_adjusted.to_excel(writer, sheet_name="mode_summary_adjusted", index=False)
    mode_pair_summary_adjusted.to_excel(writer, sheet_name="mode_pair_summary_adj", index=False)
    top_gaps_adjusted_df.to_excel(writer, sheet_name="top_gaps_adjusted", index=False)
    missing_adjusted_ticket_df.to_excel(writer, sheet_name="missing_tickets", index=False)
    adjusted_step12_wide.to_excel(writer, sheet_name="ticket_level_adjusted", index=False)
    raw_step12_wide.to_excel(writer, sheet_name="ticket_level_raw", index=False)
    figure_manifest_df.to_excel(writer, sheet_name="figure_manifest", index=False)

print(f"Saved Excel workbook: {EXCEL_PATH}")
figure_manifest_df